<a href="https://colab.research.google.com/github/Venu-max/NASSCOM-AI/blob/main/Day3_U8_%E2%80%94_Data_Cleaning_%26_Preprocessing_(Part_1)_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.2


In [2]:
# -----------------------------------------------------------
# A DELIBERATELY MESSY DATASET (so the lab is self-contained)
# -----------------------------------------------------------
# Problems baked in: missing values, disguised missing ('N/A', -1),
# duplicate rows, a number stored as text, a date as text,
# an extreme outlier, and inconsistent city spellings.
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})

In [ ]:
##1. Profile the data — find the problems

In [3]:
# -----------------------------------------------------------
# 🔹 1A. A FEW COMMANDS REVEAL MOST PROBLEMS
# -----------------------------------------------------------

df = raw.copy()        # always work on a copy
df.info()              # types + non-null counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [4]:
# -----------------------------------------------------------
# 🔹 1B. MISSING COUNTS, DUPLICATES & RANGES
# -----------------------------------------------------------

print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")

Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


In [ ]:
#LAB EXERCISE 1 — Profile it yourself
#Print the number of duplicate rows.
#Print the missing count per column.
#In a comment, list at least three problems you can already see in the data.

In [5]:
# 1. duplicate row count
print("Duplicate rows:", df.duplicated().sum())

# 2. missing per column
print("\nMissing values per column:")
print(df.isna().sum())

Duplicate rows: 1

Missing values per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [ ]:
#2. Missing values — detect & handle

In [6]:
# -----------------------------------------------------------
# 🔹 2A. UNMASK DISGUISED MISSING VALUES
# -----------------------------------------------------------

# 'N/A' (in spend) and -1 (in age) are really missing -> make them NaN
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())

Missing after unmasking:
age      2
spend    1
dtype: int64


In [7]:
# -----------------------------------------------------------
# 🔹 2B. HANDLE THE GAPS (impute)
# -----------------------------------------------------------

# median is robust to skew/outliers -> good for 'spend' and 'age'
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())

Missing after imputing: 0


In [ ]:
#LAB EXERCISE 2 — Compare drop vs impute
#Start from a fresh messy copy (ex = raw.copy()):
#Convert spend to numeric and age's -1 to NaN (as in 2A).
#Make a dropna version and a median-impute version.
#Print the row count of each — how many rows did dropping cost you?

In [8]:
# Create a fresh copy of the original data
ex = raw.copy()

# 1. Unmask missing values
# Convert 'spend' to numeric (invalid values become NaN)
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')

# Replace -1 in 'age' with NaN
ex['age'] = ex['age'].replace(-1, np.nan)

# 2a. Create a dropna version
drop_df = ex.dropna()

# 2b. Create a median-impute version
impute_df = ex.copy()
impute_df['age'] = impute_df['age'].fillna(impute_df['age'].median())
impute_df['spend'] = impute_df['spend'].fillna(impute_df['spend'].median())

# 3. Compare row counts
print("Original rows:", len(ex))
print("Rows after dropna():", len(drop_df))
print("Rows after median imputation:", len(impute_df))
print("Rows lost due to dropping:", len(ex) - len(drop_df))

Original rows: 8
Rows after dropna(): 5
Rows after median imputation: 8
Rows lost due to dropping: 3


In [ ]:
##3. Duplicates & data types

In [9]:
# -----------------------------------------------------------
# 🔹 3A. DROP DUPLICATE ROWS
# -----------------------------------------------------------

print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')

Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row


In [10]:

# -----------------------------------------------------------
# 🔹 3B. FIX DATA TYPES
# -----------------------------------------------------------

# 'date' is text -> convert to real datetimes so sorting/maths work
df['date'] = pd.to_datetime(df['date'])
# 'city' is a category -> mark it as such (saves memory, signals intent)
df['city'] = df['city'].astype('string')
print(df.dtypes)

id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [11]:
##LAB EXERCISE 3 — Dedupe & retype
#On a fresh copy ex = raw.copy():
#Convert spend to numeric and date to datetime.
#Drop duplicate rows.
#Print .dtypes and the final .shape
# Create a fresh copy of the original data
ex = raw.copy()

# 1. Fix data types: spend -> numeric, date -> datetime
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['date'] = pd.to_datetime(ex['date'])

# 2. Drop duplicate rows
ex = ex.drop_duplicates()

# 3. Print data types and final shape
print("Data Types:")
print(ex.dtypes)

print("\nFinal Shape:")
print(ex.shape)

Data Types:
id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object

Final Shape:
(7, 6)


In [ ]:
##4. Outliers — detect with the IQR rule

In [12]:
# -----------------------------------------------------------
# 🔹 4A. THE IQR RULE
# -----------------------------------------------------------

# spend has a 99000 value among ~100s -> a clear outlier
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

Q1=115.2  Q3=175.1  IQR=59.8
Normal range: 25.5 to 264.9

Outlier rows:
  name    spend
5  Fin  99000.0


In [13]:
# -----------------------------------------------------------
# 🔹 4B. ONE WAY TO TREAT THEM — CAP (winsorise)
# -----------------------------------------------------------

# clip values to the IQR bounds instead of deleting the row
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       264.875
6  Gus    110.0       110.000


In [14]:
# LAB #EXERCISE 4 — Find the outliers
#Using the age column of the cleaned df:
#Compute Q1, Q3 and the IQR.
#Compute the lower & upper bounds (Q1 − 1.5·IQR, Q3 + 1.5·IQR).
#Print any rows whose age falls outside those bounds (there may be none — that's a valid result).
# 1. Q1, Q3, IQR for 'age'
Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)
IQR = Q3 - Q1

print("Q1 =", Q1)
print("Q3 =", Q3)
print("IQR =", IQR)

# 2. Lower & upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower Bound =", lower_bound)
print("Upper Bound =", upper_bound)

# 3. Rows outside the bounds
outliers = df[(df['age'] < lower_bound) | (df['age'] > upper_bound)]

print("\nOutlier rows:")
print(outliers)

Q1 = 29.25
Q3 = 34.0
IQR = 4.75
Lower Bound = 22.125
Upper Bound = 41.125

Outlier rows:
Empty DataFrame
Columns: [id, name, age, city, spend, date, spend_capped]
Index: []


In [ ]:
#5. Messy text & inconsistent categories

In [15]:

# -----------------------------------------------------------
# 🔹 5A. THE PROBLEM — ONE CITY, MANY SPELLINGS
# -----------------------------------------------------------

print(df['city'].value_counts())   # ' Pune ', 'pune', 'Pune.' all look different!


city
 Pune     1
pune      1
DELHI     1
Delhi     1
Mumbai    1
bombay    1
Pune.     1
Name: count, dtype: Int64


In [16]:
# -----------------------------------------------------------
# 🔹 5B. STANDARDISE THE STRINGS
# -----------------------------------------------------------

s = df['city'].astype('string')
s = s.str.strip()                       # trim whitespace
s = s.str.lower()                       # unify case
s = s.str.replace('.', '', regex=False) # drop stray punctuation
s = s.replace({'bombay': 'mumbai'})     # map known variants to one label
df['city'] = s
print(df['city'].value_counts())        # now clean categories

city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


In [17]:
##LAB EXERCISE 5 — Clean a messy column
#The messy series below has the same problems.
#Trim whitespace and lowercase it.
#Map the variant 'n.y.' to 'new york'.
#Print value_counts() — you should end up with just two clean categories.
import pandas as pd

messy = pd.Series([' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
                  dtype='string')

# 1. strip + lower
messy = messy.str.strip().str.lower()

# 2. map 'n.y.' -> 'new york' (after lowering)
messy = messy.replace({'n.y.': 'new york'})

# 3. value_counts()
print(messy.value_counts())

london      3
new york    3
Name: count, dtype: Int64


In [18]:
##The cleaned dataset
# After all the steps above, here's the cleaned frame
clean = df.drop(columns=['spend_capped'])
print('Final shape:', clean.shape)
print('Missing values:', int(clean.isna().sum().sum()))
print('Duplicates    :', int(clean.duplicated().sum()))
clean

Final shape: (7, 6)
Missing values: 0
Duplicates    : 0


,id,name,age,city,spend,date
0,1,Ana,30.0,pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,29.5,delhi,200.2,2024-01-07
3,4,Di,41.0,delhi,120.5,2024-01-08
4,5,Eve,29.5,mumbai,150.0,2024-01-09
5,6,Fin,38.0,mumbai,99000.0,2024-01-10
6,7,Gus,29.0,pune,110.0,2024-01-11
